# Fine tuning distilbert

This notebook can be used to fine tune distilbert on GoEmotions for use within A2F. A similar script can be found in the testdistil.py file, again for fine tuning distilbert.

## Set up

In [1]:
# install libraries
!pyenv install 3.13
!pyenv global 3.13
!pyenv local 3.13
!pip install datasets
!pip install torch torchvision transformers

:: [Info] ::  Mirror: https://www.python.org/ftp/python
:: [Info] ::  Mirror: https://downloads.python.org/pypy/versions.json
:: [Info] ::  Mirror: https://api.github.com/repos/oracle/graalpython/releases
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached filelock-3.32.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached numpy-2.5.1-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached pyarrow-25.0.0-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached pandas-3.0.5-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.69.1-py3-none-any.whl.metadata (57 kB)
  Using cached xxhash-3.8.1-cp313-cp313-win_amd64.whl.metadata (15 kB)
  Using cached multiprocess-0.70.19-py313-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadat

  Using cached torch-2.13.0-cp313-cp313-win_amd64.whl.metadata (39 kB)
  Using cached torchvision-0.28.0-cp313-cp313-win_amd64.whl.metadata (5.6 kB)
  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached setuptools-83.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached pillow-12.3.0-cp313-cp313-win_amd64.whl.metadata (9.3 kB)
  Using cached regex-2026.7.19-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer-0.27.0-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-win_amd64.whl.metadata (2.8 kB)
  Using cached shellingha

In [ ]:
# Import libraries
from datasets import load_dataset, Dataset, DatasetDict
import pandas as pd
import torch
import torchvision
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import classification_report
import numpy as np
import os

In [ ]:
# check gpu is available
if torch.cuda.is_available():
  print("GPU is available")
else:
  print("GPU is not available")

# Base directory
BASE_DIR = os.path.dirname(os.path.abspath(__file__))   

GPU is available


In [ ]:
# Import Go-Emotions
emotions_db = load_dataset("mrm8488/goemotions")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/7.11k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


goemotions.csv: reconstructing file:   0%|          |  0.00B / 42.7MB            

goemotions.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/211225 [00:00<?, ? examples/s]

In [ ]:
# Inspect dataset
emotions_db.set_format(type="pandas")


## Pre processing

In [ ]:
# Audio2Face emotions
columns =  [
    'amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral'
]

# Custom dataframe
custom_df = pd.DataFrame(columns = columns)
train_db = emotions_db['train'].to_pandas()

# Emotion columns
custom_df['amazement'] = train_db['amusement'] + train_db['realization'] + train_db['surprise'] + train_db['admiration']
custom_df['anger'] = train_db['anger'] + train_db['annoyance'] + train_db['disapproval']
custom_df['cheekiness'] = train_db['caring'] + train_db['love'] + train_db['desire']
custom_df['disgust'] = train_db['disgust'] + train_db['remorse']
custom_df['fear'] = train_db['fear'] + train_db['nervousness']
custom_df['grief'] = train_db['grief']
custom_df['joy'] = train_db['joy'] + train_db['pride'] + train_db['optimism'] + train_db['gratitude'] + train_db['relief'] + train_db['excitement']
custom_df['out of breath'] = train_db['confusion']
custom_df['pain'] = train_db['disappointment']
custom_df['sadness'] = train_db['sadness'] + train_db['embarrassment']
custom_df['neutral'] = train_db['neutral'] + train_db['approval'] + train_db['curiosity']

# Text column
custom_df['text'] = train_db['text']

# Remove columns with all elements zero
custom_df = custom_df[(custom_df.T != 0).any()]
custom_dataset = Dataset.from_pandas(custom_df)

In [ ]:
# Load tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Pre processing
def pre_process(examples):
  return tokenizer(examples['text'][0], truncation = True, padding = 'max_length')

# Apply tokenization with batches
distil_dataset = custom_dataset.map(pre_process, batch_size = 10)

Map:   0%|          | 0/211225 [00:00<?, ? examples/s]

## Format the dataset for the trainer

In [ ]:
def add_label(example):
  example['label'] = [example[col] for col in columns]
  return example

distil_dataset = distil_dataset.map(add_label)

Map:   0%|          | 0/211225 [00:00<?, ? examples/s]

In [ ]:
print(distil_dataset)

Dataset({
    features: ['amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral', 'text', 'input_ids', 'attention_mask', 'label'],
    num_rows: 211225
})


## Split into train, test and evaluation data

In [ ]:
ds_train_devtest = distil_dataset.train_test_split(test_size=0.2, seed=42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size=0.5, seed=42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'eval': ds_devtest['train'],
    'test': ds_devtest['test']
})
print(ds_splits)


DatasetDict({
    train: Dataset({
        features: ['amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral', 'text', 'input_ids', 'attention_mask', 'label'],
        num_rows: 168980
    })
    eval: Dataset({
        features: ['amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral', 'text', 'input_ids', 'attention_mask', 'label'],
        num_rows: 21122
    })
    test: Dataset({
        features: ['amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral', 'text', 'input_ids', 'attention_mask', 'label'],
        num_rows: 21123
    })
})


## Fine tuning

In [ ]:
# Load DistilBERT model for classification
'amazement', 'anger', 'cheekiness', 'disgust', 'fear', 'grief', 'joy', 'out of breath', 'pain', 'sadness', 'neutral'
id2label = {
    "0": "amazement",
    "1": "anger",
    "2": "cheekiness",
    "3": "disgust",
    "4": "fear",
    "5": "grief",
    "6": "joy",
    "7": "out of breath",
    "8": "pain",
    "9": "sadness",
    "10": "neutral"
}
label2id = {
    "amazement" : 0,
    "anger" : 1,
    "cheekiness" : 2,
    "disgust" : 3,
    "fear" : 4,
    "grief" : 5,
    "joy" : 6,
    "out of breath" : 7,
    "pain" : 8,
    "sadness" : 9,
    "neutral" : 10
}
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=11, id2label=id2label, label2id=label2id)

# Setting up training settings
batch_size = 64
training_args = TrainingArguments(
    output_dir="./results",          # Directory for saving results
    learning_rate=2e-5,              # Initial learning rate
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    evaluation_strategy="epoch",
    num_train_epochs=2,              # Number of epochs
    weight_decay=0.01,               # Regularization
    logging_dir="./logs",            # Directory for logs
    logging_steps=10,                 # Log every 10 steps
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
trainer = Trainer(
    model = model,                             # The DistilBERT model
    args = training_args,                      # Training arguments
    train_dataset = ds_splits['train'],        # Training data
    eval_dataset= ds_splits['eval'],           # Validation data
    tokenizer = tokenizer
)

# Start training
trainer.train()

Step,Training Loss
10,0.569000
20,0.397922
30,0.334234
40,0.303167
50,0.298197
60,0.291714
70,0.276334
80,0.305201
90,0.258610
100,0.274316


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

## Evaluation

In [ ]:
# Evaluate general performance
predictions = trainer.predict(ds_splits['test'])
preds = np.argmax(predictions.predictions, axis = 1)
actual_labels = ds_splits['test']['labels']
print(classification_report(actual_labels, preds))

# Error analysis
for idx, (actual, pred) in enumerate(zip(actual_labels, preds)):
  print(f"Example {idx}: \n")
  print(f"Sentence: {ds_splits['test']['text'][idx]} \n")
  print(f"Actual: { actual} \n")
  print(f"Predicted {predicted} \n")
  print("\n")


## Deployment

In [ ]:
# Save the model and tokenizer
model.save_pretrained(os.path.join(BASE_DIR, "../../../data/fine_tuned_model"))
tokenizer.save_pretrained(os.path.join(BASE_DIR, "../../../data/fine_tuned_model"))